# Lekcija 26: Nastanak slike

Dosad smo obrađivali slike bez razmišljanja o tome odakle dolaze. Ova lekcija razjašnjava tu zagonetku razmatrajući **geometriju** (kako se projekcijska matrica izvodi iz modela kamere s malim otvorom), **optiku** (što stvarna leća radi, a model kamere s malim otvorom ne opisuje &mdash; izoštravanje i zamućenje) te **senzor i boju** (kako stvarni senzor bilježi samo jednu boju po pikselu i mora rekonstruirati ostale, uz dvije klasične zamke &mdash; gamu i ravnotežu bijele boje &mdash; koje otežavaju naivnu obradu slike).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Izvod projekcijske matrice kamere s malim otvorom

Kamera s malim otvorom propušta samo zrake svjetlosti koje prolaze kroz određenu točku (*središte projekcije*). Prema sličnosti trokuta, 3D točka $(X, Y, Z)$ (u koordinatama sa središtem u kameri) pada na ravninu slike na položaj

$$x = f\frac{X}{Z}, \qquad y = f\frac{Y}{Z}$$

gdje je $f$ udaljenost od otvora do ravnine slike. Pretvorba u pikselne jedinice (uz uvažavanje veličine piksela i položaja ishodišta) zahtijeva **unutarnje parametre** $K$, a uključivanje položaja i orijentacije kamere u odnosu na svijet zahtijeva **vanjske parametre** $[R | t]$. Zajedno, u homogenim koordinatama:

$$\underbrace{\begin{bmatrix}u\\v\\1\end{bmatrix}}_{\text{pixel}} \;\propto\; \underbrace{\begin{bmatrix}f_x&0&c_x\\0&f_y&c_y\\0&0&1\end{bmatrix}}_{K}\underbrace{\begin{bmatrix}R & t\end{bmatrix}}_{\text{extrinsics}}\begin{bmatrix}X\\Y\\Z\\1\end{bmatrix}_{\text{world}}$$

To je upravo $P = K [R | t]$ koji će se koristiti u lekcijama 27–29 (Kalibracija kamere, Epipolarna geometrija i Struktura iz gibanja). Svaki simbol može se povezati s fizičkim uzrokom: $f_x, f_y$ sa žarišnom duljinom i veličinom piksela; $(c_x, c_y)$ s mjestom gdje optička os siječe senzor (poznatim kao *glavna točka*); $R, t$ s položajem i orijentacijom kamere.

In [ ]:
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)
R, t = np.eye(3), np.array([0, 0, 5.0])
P = K @ np.hstack([R, t.reshape(3, 1)])

cube_corners = np.array([[x, y, z] for x in (-1, 1) for y in (-1, 1) for z in (-1, 1)], dtype=np.float64)
homogeneous = np.hstack([cube_corners, np.ones((8, 1))])
projected = (P @ homogeneous.T).T
pixels = projected[:, :2] / projected[:, 2:3]

edges = [(0, 1), (0, 2), (0, 4), (1, 3), (1, 5), (2, 3), (2, 6),
         (3, 7), (4, 5), (4, 6), (5, 7), (6, 7)]

plt.figure(figsize=(5, 4))
for i, j in edges:
    plt.plot([pixels[i, 0], pixels[j, 0]], [pixels[i, 1], pixels[j, 1]], color='tab:blue')
plt.scatter(*pixels.T, color='red', zorder=5)
plt.gca().invert_yaxis()
plt.title('A 3D cube, projected through P = K[R|t]')
plt.show()

## Stvarne leće: fokus i zamućenje zbog defokusiranja

Kamera s malim otvorom propušta toliko malo svjetlosti da zahtijeva iznimno duge ekspozicije. Zato stvarne kamere koriste leću za prikupljanje mnogo više svjetlosti. Međutim, leće oštro izoštravaju samo objekte na određenoj udaljenosti (u **žarišnoj ravnini**). Točke bliže ili dalje od nje rasprostiru svoju svjetlost po malom disku na senzoru (**krugu zamućenja**) umjesto po jednoj točki &mdash; upravo kao konvolucijske jezgre zamućenja iz lekcija 9–10, samo s veličinom jezgre koja ovisi o dubini.

In [ ]:
scene = np.zeros((200, 300, 3), dtype=np.uint8)
cv2.rectangle(scene, (20, 20), (90, 180), (255, 120, 30), -1)    # near object
cv2.rectangle(scene, (110, 20), (190, 180), (30, 200, 255), -1)  # object at the focal plane
cv2.rectangle(scene, (210, 20), (280, 180), (120, 255, 60), -1)  # far object

depths_m = {'near': 2.0, 'mid': 5.0, 'far': 9.0}
focus_depth_m = 5.0

def defocus_kernel_size(depth, focus, strength=15.0):
    radius = strength * abs(depth - focus) / focus
    ksize = int(radius) * 2 + 1
    return max(1, ksize)

def blur_with_context(image, y0, y1, x0, x1, ksize, pad):
    """Blur a region using a padded crop, so the blur can pull in real neighboring
    pixels (including the background) instead of only the region's own solid color."""
    if ksize <= 1:
        return image[y0:y1, x0:x1].copy()
    yy0, yy1 = max(0, y0 - pad), min(image.shape[0], y1 + pad)
    xx0, xx1 = max(0, x0 - pad), min(image.shape[1], x1 + pad)
    blurred = cv2.GaussianBlur(image[yy0:yy1, xx0:xx1], (ksize, ksize), 0)
    return blurred[y0 - yy0:y0 - yy0 + (y1 - y0), x0 - xx0:x0 - xx0 + (x1 - x0)]

defocused = scene.copy()
regions = {'near': (20, 180, 20, 90), 'mid': (20, 180, 110, 190), 'far': (20, 180, 210, 280)}
for name, (y0, y1, x0, x1) in regions.items():
    k = defocus_kernel_size(depths_m[name], focus_depth_m)
    print(f'{name:>5} object: depth={depths_m[name]}m, blur kernel size={k}')
    defocused[y0:y1, x0:x1] = blur_with_context(scene, y0, y1, x0, x1, k, pad=k)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
axes[0].set_title('All in focus (pinhole idealization)')
axes[1].imshow(cv2.cvtColor(defocused, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Focused at {focus_depth_m}m\n(near/far objects blurred)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

To objašnjava zašto širi otvor blende prikuplja više svjetlosti (uz manju dubinsku oštrinu), dok uži otvor daje oštru sliku (ali propušta manje svjetlosti) &mdash; klasičan fotografski kompromis između otvora blende i dubinske oštrine. Zato je na portretnim fotografijama pozadina često zamućena, a subjekt oštar.

## Bayerovo polje filtara boje: jedna boja po pikselu

Radi smanjenja troškova, većina senzora kamera ne mjeri zasebno crvenu, zelenu i plavu na svakom pikselu. Umjesto toga, jedno polje monokromatskih fotosenzora nalazi se ispod mozaika sitnih filtara boje, pa svaki pojedini piksel fizički bilježi samo *jednu* od triju boja. Najčešći raspored (RGGB) &mdash; **Bayerov filtar** (<a href="../references.html#bayer-1976">Bayer, 1976.</a>) &mdash; ponavlja se u blokovima 2x2: jedan crveni, jedan plavi i *dva* zelena filtra, jer je ljudsko oko najosjetljivije na zelenu. Stoga svakom pikselu sirovog izlaza senzora već po konstrukciji nedostaju dvije trećine informacija o boji.

In [ ]:
rng = np.random.default_rng(0)
bayer_scene = np.zeros((120, 160, 3), dtype=np.uint8)
cv2.rectangle(bayer_scene, (10, 10), (70, 100), (40, 180, 220), -1)
cv2.circle(bayer_scene, (110, 60), 40, (200, 90, 40), -1)
bayer_scene = np.clip(bayer_scene.astype(np.float64) + rng.normal(0, 4, bayer_scene.shape), 0, 255).astype(np.uint8)

def make_bayer_rggb(img):
    # (row, col) parity: (even,even)=R, (even,odd)=G, (odd,even)=G, (odd,odd)=B
    H, W = img.shape[:2]
    mosaic = np.zeros((H, W), dtype=np.uint8)
    mosaic[0::2, 0::2] = img[0::2, 0::2, 2]  # R  (img is BGR, so channel 2 = R)
    mosaic[0::2, 1::2] = img[0::2, 1::2, 1]  # G
    mosaic[1::2, 0::2] = img[1::2, 0::2, 1]  # G
    mosaic[1::2, 1::2] = img[1::2, 1::2, 0]  # B
    return mosaic

def colorize_bayer_rggb(mosaic):
    """Put each pixel's raw reading into its OWN filter's color channel, zeroing the
    other two -- this is what actually lets you see the RGGB pattern, since a plain
    grayscale view of the mosaic just shows intensities with no color information at all."""
    H, W = mosaic.shape
    colored = np.zeros((H, W, 3), dtype=np.uint8)
    colored[0::2, 0::2, 0] = mosaic[0::2, 0::2]  # R filter
    colored[0::2, 1::2, 1] = mosaic[0::2, 1::2]  # G filter
    colored[1::2, 0::2, 1] = mosaic[1::2, 0::2]  # G filter
    colored[1::2, 1::2, 2] = mosaic[1::2, 1::2]  # B filter
    return colored

bayer = make_bayer_rggb(bayer_scene)
bayer_colored = colorize_bayer_rggb(bayer)

y0, y1, x0, x1 = 40, 56, 20, 36  # a 16x16 crop, fully inside a colored shape

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(cv2.cvtColor(bayer_scene, cv2.COLOR_BGR2RGB))
axes[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor='white', facecolor='none', linewidth=1.5))
axes[0].set_title('true scene (each pixel: R, G, and B)')
axes[1].imshow(bayer[y0:y1, x0:x1], cmap='gray')
axes[1].set_title('raw mosaic\n(grayscale readout -- no color info visible)')
axes[2].imshow(bayer_colored[y0:y1, x0:x1])
axes[2].set_title('same patch, colorized by filter\n(the actual RGGB pattern)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Demosaiciranje: rekonstrukcija RGB-a iz mozaika

**Demosaiciranje** (ili uklanjanje Bayerova mozaika) popunjava dvije nedostajuće boje svakog piksela interpolacijom iz susjeda iste boje — najjednostavnija je varijanta bilinearna: prosjek obližnjih poznatih uzoraka boje ponderiran udaljenošću, upravo kao bilinearna interpolacija iz lekcije 9, samo što se interpolira preko rijetke, pravilno raspoređene mreže poznatih uzoraka umjesto preko guste slike smanjene rezolucije.

In [ ]:
def demosaick_bilinear_rggb(mosaic):
    H, W = mosaic.shape
    R = np.zeros((H, W)); G = np.zeros((H, W)); B = np.zeros((H, W))
    R[0::2, 0::2] = mosaic[0::2, 0::2]
    G[0::2, 1::2] = mosaic[0::2, 1::2]; G[1::2, 0::2] = mosaic[1::2, 0::2]
    B[1::2, 1::2] = mosaic[1::2, 1::2]

    Rmask = np.zeros((H, W)); Rmask[0::2, 0::2] = 1
    Gmask = np.zeros((H, W)); Gmask[0::2, 1::2] = 1; Gmask[1::2, 0::2] = 1
    Bmask = np.zeros((H, W)); Bmask[1::2, 1::2] = 1

    def fill(channel, mask):
        # weighted average of known same-color neighbors within a 3x3 window
        kernel = np.array([[0.25, 0.5, 0.25], [0.5, 1.0, 0.5], [0.25, 0.5, 0.25]])
        weight_sum = cv2.filter2D(mask, -1, kernel, borderType=cv2.BORDER_REFLECT)
        value_sum = cv2.filter2D(channel * mask, -1, kernel, borderType=cv2.BORDER_REFLECT)
        filled = channel.copy()
        empty = mask == 0
        filled[empty] = value_sum[empty] / np.clip(weight_sum[empty], 1e-6, None)
        return filled

    R_full, G_full, B_full = fill(R, Rmask), fill(G, Gmask), fill(B, Bmask)
    return np.clip(np.stack([B_full, G_full, R_full], axis=-1), 0, 255).astype(np.uint8)

manual_demosaick = demosaick_bilinear_rggb(bayer)
cv2_demosaick = cv2.cvtColor(bayer, cv2.COLOR_BayerBG2BGR)

manual_err = np.abs(manual_demosaick.astype(np.float64) - bayer_scene.astype(np.float64)).mean()
cv2_err = np.abs(cv2_demosaick.astype(np.float64) - bayer_scene.astype(np.float64)).mean()
print(f'manual bilinear demosaick MAE vs ground truth: {manual_err:.2f}')
print(f'cv2 demosaick MAE vs ground truth:              {cv2_err:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [bayer_scene, manual_demosaick, cv2_demosaick],
                          ['ground truth', 'manual bilinear\ndemosaick', 'cv2 demosaick']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Obje metode dobro rekonstruiraju referentnu sliku (preostala pogreška uglavnom je šum senzora već prisutan u mozaiku, a ne artefakt demosaiciranja) — bilinearno popunjavanje implementirano od početka i ugrađeno OpenCV-jevo demosaiciranje razlikuju se za djelić vrijednosti piksela. Stvarni algoritmi demosaiciranja složeniji su od čiste bilinearne interpolacije (interpolacija koja uvažava rubove izbjegava zamućenje preko granica objekata, jer upravo naivno bilinearno demosaiciranje uzrokuje obojene artefakte nalik „patentnom zatvaraču” duž oštrih rubova na jeftinim kamerama). No temeljna ideja — interpolirati svaku nedostajuću boju iz najbližih susjeda iste boje — ostaje ista.

### Zamka: pogrešan uzorak mozaika

Raspored R/G/B je važan, a postoji više konvencija (RGGB, BGGR, GRBG, GBRG, ovisno o senzoru). Demosaiciranje uz *pogrešno* pretpostavljen uzorak ne završava očitom pogreškom — tiho stvara uvjerljivu sliku s izrazito pogrešnim bojama, jer svaki piksel dobije vrijednost, samo pogrešnu.

In [ ]:
wrong_demosaick = cv2.cvtColor(bayer, cv2.COLOR_BayerRG2BGR)  # wrong pattern assumption
wrong_err = np.abs(wrong_demosaick.astype(np.float64) - bayer_scene.astype(np.float64)).mean()
print(f'correct pattern (BayerBG2BGR) MAE: {cv2_err:.2f}')
print(f'WRONG pattern (BayerRG2BGR) MAE:   {wrong_err:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(cv2.cvtColor(cv2_demosaick, cv2.COLOR_BGR2RGB)); axes[0].set_title('correct pattern', fontsize=9)
axes[1].imshow(cv2.cvtColor(wrong_demosaick, cv2.COLOR_BGR2RGB)); axes[1].set_title('wrong pattern', fontsize=9)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Pogreška poraste za više od jednog reda veličine ako se koristi pogrešan uzorak &mdash; svaki crveni uzorak tretira se kao plavi i obratno, što daje sliku s izrazito pomaknutim bojama. Zato postupci obrade RAW slika uvijek moraju znati točan raspored senzora kamere.

## Gama korekcija: vrijednosti piksela nisu linearna svjetlost

Kamere i zasloni ne pohranjuju svjetlinu proporcionalno fizičkom intenzitetu svjetlosti (**linearnoj radijanciji**). Umjesto toga, vrijednosti su **gama-kodirane**: `encoded = linear ** (1/gamma)` (obično gama &asymp; 2.2, blizu standarda sRGB), čime se više kodiranih razina dodjeljuje tamnijim tonovima &mdash; u skladu s većom osjetljivošću ljudskog vida na sjene nego na svijetle dijelove te, povijesno, s odzivom CRT zaslona na napon. Dekodiranje to obrće: `linear = encoded ** gamma`.

In [ ]:
def encode_gamma(linear, gamma=1 / 2.2):
    return np.clip(linear, 0, 1) ** gamma

def decode_gamma(encoded, gamma=2.2):
    return np.clip(encoded, 0, 1) ** gamma

linear_values = np.linspace(0, 1, 100)
plt.plot(linear_values, linear_values, '--', color='gray', label='identity (no gamma)')
plt.plot(linear_values, encode_gamma(linear_values), label='gamma-encoded (stored pixel value)')
plt.xlabel('linear light intensity')
plt.ylabel('encoded value')
plt.legend(fontsize=8)
plt.title('Gamma encoding curve')
plt.show()

### Zamka: usrednjavanje kodiranih piksela nije usrednjavanje svjetlosti

Miješanje, promjena veličine i zamućivanje (lekcije 8–10) prešutno pretpostavljaju da se usrednjava linearna veličina. Vrijednosti piksela obično nisu linearne &mdash; gama-kodirane su &mdash; pa naivno usrednjavanje dviju kodiranih vrijednosti piksela daje *pogrešnu fizičku svjetlinu*, ponekad vrlo izraženo.

In [ ]:
black_encoded, white_encoded = encode_gamma(0.0), encode_gamma(1.0)

# WRONG: average the encoded (stored) pixel values directly
naive_blend_encoded = (black_encoded + white_encoded) / 2
naive_blend_as_linear_light = decode_gamma(naive_blend_encoded)

# RIGHT: decode to linear light first, average there, then re-encode for storage/display
correct_blend_linear = (decode_gamma(black_encoded) + decode_gamma(white_encoded)) / 2
correct_blend_encoded = encode_gamma(correct_blend_linear)

print(f'true 50% linear-light gray, correctly encoded: {correct_blend_encoded:.3f}')
print(f'naive average of encoded black/white:           {naive_blend_encoded:.3f}')
print()
print(f'naive result represents only {naive_blend_as_linear_light:.1%} of the true linear brightness '
      f'(should be 50%)')

swatch = np.zeros((80, 240), dtype=np.float64)
swatch[:, :80] = naive_blend_encoded
swatch[:, 80:160] = correct_blend_encoded
swatch[:, 160:] = 0.5
plt.imshow(swatch, cmap='gray', vmin=0, vmax=1)
plt.title('naive blend  |  correct blend  |  encoded 0.5 for reference')
plt.axis('off')
plt.show()

Naivna mješavina vidljivo je i izrazito tamnija od fizički ispravnog rezultata s 50 % sive &mdash; stvarna i česta pogreška kada kod za obradu slike (zamućivanje, izradu mipmapa, alfa miješanje) radi izravno s gama-kodiranim vrijednostima piksela umjesto da ih najprije linearizira.

## Ravnoteža bijele boje: boja ovisi o izvoru svjetlosti

Kamera bilježi *umnožak* reflektancije površine i boje izvora osvjetljenja, a ne samo „pravu” boju površine &mdash; bijeli list papira izgleda narančasto pod toplim svjetlom volframove žarulje, a plavkasto pod oblačnim nebom, iako naš vidni sustav to obično (uglavnom) automatski kompenzira (postojanost boje). Kamere moraju namjerno provesti istu korekciju: **uravnoteženje bijele boje**.

In [ ]:
rng = np.random.default_rng(0)
true_scene = np.zeros((150, 150, 3), dtype=np.uint8)
cv2.rectangle(true_scene, (20, 20), (130, 130), (150, 150, 150), -1)  # a neutral gray card
cv2.circle(true_scene, (75, 75), 30, (60, 120, 200), -1)
true_scene = np.clip(true_scene.astype(np.float64) + rng.normal(0, 5, true_scene.shape), 0, 255).astype(np.uint8)

tungsten_gains = np.array([1.3, 1.0, 0.6])  # BGR: boosts red, cuts blue -- a warm cast
cast_scene = np.clip(true_scene.astype(np.float64) * tungsten_gains, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(cv2.cvtColor(true_scene, cv2.COLOR_BGR2RGB))
axes[0].set_title('True colors')
axes[1].imshow(cv2.cvtColor(cast_scene, cv2.COLOR_BGR2RGB))
axes[1].set_title('Under warm tungsten light')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Pretpostavka sivog svijeta

Jednostavan, klasičan algoritam uravnoteženja bijele boje pretpostavlja da se boje, usrednjene preko cijelog stvarnog prizora, približno ponište do neutralne sive. Uz tu pretpostavku, svaka sustavna razlika između prosjeka kanala fotografije otkriva obojenost izvora osvjetljenja &mdash; pa bi skaliranje svakog kanala radi izjednačavanja prosjeka trebalo približno ukloniti taj utjecaj.

In [ ]:
def gray_world_white_balance(image):
    image_f = image.astype(np.float64)
    channel_means = image_f.reshape(-1, 3).mean(axis=0)
    target_gray = channel_means.mean()
    gains = target_gray / channel_means
    return np.clip(image_f * gains, 0, 255).astype(np.uint8)

corrected_scene = gray_world_white_balance(cast_scene)

error_before = np.abs(cast_scene.astype(int) - true_scene.astype(int)).mean()
error_after = np.abs(corrected_scene.astype(int) - true_scene.astype(int)).mean()
print(f'mean abs pixel error before correction: {error_before:.2f}')
print(f'mean abs pixel error after correction:  {error_after:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [true_scene, cast_scene, corrected_scene],
                          ['True colors', 'Color cast', 'Gray-world corrected']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Korekcija znatno smanjuje pogrešku, ali ne na nulu &mdash; pretpostavka sivog svijeta ovdje je samo približna (ovaj mali prizor nije u prosjeku savršeno neutralan jer njime dominira obojeni krug). Upravo zato stvarne kamere često kombiniraju tu pretpostavku s drugim informacijama (detektiranim sivim ili bijelim referentnim područjem, naučenim statistikama prizora ili korisnički odabranim postavkama) umjesto da se oslanjaju samo na nju.

### Zadaci

1. Promijenite `strength` u `defocus_kernel_size` kako biste dobili mnogo manju dubinsku oštrinu (veći `strength`) ili mnogo veću (manji `strength`). Što bi se trebalo dogoditi pri `strength=0` i daje li kod takav rezultat?
2. Izmijenite `make_bayer_rggb` i dodjele `Rmask`/`Gmask`/`Bmask` u `demosaick_bilinear_rggb` kako biste izgradili i dekodirali mozaik BGGR umjesto RGGB (zamijenite kutove s crvenom i plavom bojom). Pronađite odgovarajući OpenCV kod isprobavanjem svakog od `cv2.COLOR_BayerBG2BGR`, `cv2.COLOR_BayerGB2BGR`, `cv2.COLOR_BayerGR2BGR`, `cv2.COLOR_BayerRG2BGR` i provjerom koji vraća MAE na razinu šuma — OpenCV-jevu konvenciju imenovanja tih kodova vrlo je lako obrnuti, što je i samo mali primjer poante ove lekcije o tihim pogreškama zbog nepodudaranja uzorka.
3. Ponovite eksperiment gama-miješanja, ali miješajte `0.2` i `0.8` (umjesto potpuno crne i bijele) i naivnim i ispravnim načinom. Je li razlika između njih veća ili manja nego u slučaju crne i bijele, i zašto bi to bilo očekivano s obzirom na oblik gama-krivulje?
4. Izmijenite demonstraciju ravnoteže bijele boje tako da prizorom dominira veliko *zasićeno crveno* područje umjesto neutralne sive karte (dakle, učinite temeljnu pretpostavku sivog svijeta očito netočnom). Poboljšava li `gray_world_white_balance` i dalje rezultat, ne mijenja li ga ili ga aktivno pogoršava?